# NB08 — Stage A: Pipeline Search (controlled, fixed budget)

**Purpose:** find the single best *pipeline structure* for Ben-Sarc, comparing structural choices
(full-FT vs layer-freezing, ±LLRD, base vs large backbone, max_len, ±FGM, ±label-smoothing) **under one
fixed training budget** so the comparison is fair. No epsilon sweep here (that is Stage B / NB09).

Also runs a **faithful reproduction of Zihan et al. (ICECTE-2026)** recipe on our *clean* splits — labelled
separately because it uses their 15-epoch budget, not the fixed comparison budget. Its job is to **measure
the leakage gap** vs their reported 0.890.

Fixed budget for the fair set: `EPOCHS_A`, batch 32, max_len 128, FGM ε=0.5 where used.
Independent of NB07/10. Resumable. Best pipeline (by **val** macro-F1) is consumed by NB09.


In [9]:
import importlib, subprocess, sys
def ensure(p,i=None):
    try: importlib.import_module(i or p)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",p],check=False)
for pkg,imp in [("transformers","transformers"),("accelerate","accelerate"),("scikit-learn","sklearn"),
                ("pandas","pandas"),("numpy","numpy")]: ensure(pkg,imp)
print("deps ok")

deps ok


In [10]:
import os, json, time, random, shutil, warnings, unicodedata, re, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
warnings.filterwarnings("ignore")
import transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
HAS_CUDA=torch.cuda.is_available()
HAS_BF16=HAS_CUDA and torch.cuda.is_bf16_supported()
print("cuda:", HAS_CUDA, "| bf16:", HAS_BF16)
if HAS_CUDA: print("GPU:", torch.cuda.get_device_name(0),
                    f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
SEED=42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if HAS_CUDA: torch.cuda.manual_seed_all(s)
set_seed(SEED)

torch: 2.8.0+cu128 | transformers: 4.40.0
cuda: True | bf16: True
GPU: NVIDIA A40 47.7 GB


In [11]:
def find_repo_root():
    for c in [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents):
        if (c/"01_data").exists(): return c.resolve()
    raise RuntimeError("repo root not found.")
ROOT=find_repo_root(); SPLITS=ROOT/"01_data"/"interim"/"splits"
CKPT=ROOT/"03_checkpoints"; TABLES=ROOT/"04_outputs"/"tables"; PRED=ROOT/"04_outputs"/"predictions"
for p in [CKPT,TABLES,PRED]: p.mkdir(parents=True,exist_ok=True)
print("ROOT:", ROOT, "| SPLITS exists:", SPLITS.exists()); assert SPLITS.exists(), "Run NB01 first."

_ZW=dict.fromkeys(map(ord,["\u200b","\u200c","\u200d","\ufeff"]),None)
def norm_key(s):
    if not isinstance(s,str): s="" if pd.isna(s) else str(s)
    s=unicodedata.normalize("NFC",s).translate(_ZW); return re.sub(r"\s+"," ",s).strip().casefold()
def assert_no_leak(tr,va,te,col="text"):
    a={norm_key(x) for x in tr[col]};b={norm_key(x) for x in va[col]};c={norm_key(x) for x in te[col]}
    assert len(a&b)==0 and len(a&c)==0 and len(b&c)==0, "LEAK detected"; return {"tv":0,"tt":0,"vt":0}
def softmax_np(x):
    x=np.asarray(x,np.float64); x=x-x.max(1,keepdims=True); e=np.exp(x); return e/e.sum(1,keepdims=True)
def eval_from_logits(labels,logits,nl):
    labels=np.asarray(labels); preds=np.asarray(logits).argmax(-1)
    m={"accuracy":float(accuracy_score(labels,preds)),
       "macro_f1":float(f1_score(labels,preds,average="macro",zero_division=0)),
       "weighted_f1":float(f1_score(labels,preds,average="weighted",zero_division=0))}
    if nl==2: m["binary_f1"]=float(f1_score(labels,preds,average="binary",pos_label=1,zero_division=0))
    return m,preds
def make_compute_metrics():
    def _cm(ep):
        logits,labels=ep; preds=np.asarray(logits).argmax(-1)
        return {"macro_f1":f1_score(labels,preds,average="macro",zero_division=0)}
    return _cm
def save_predictions(path,texts,labels,logits,nl,extra=None):
    logits=np.asarray(logits); probs=softmax_np(logits); preds=logits.argmax(-1)
    d={"text":[str(t) for t in texts],"gold_label":np.asarray(labels),"pred_label":preds,
       "correct":(np.asarray(labels)==preds).astype(int)}
    for k in range(nl): d[f"logit_{k}"]=logits[:,k]
    for k in range(nl): d[f"prob_{k}"]=probs[:,k]
    df=pd.DataFrame(d)
    if extra:
        for k,v in extra.items(): df[k]=v
    df.to_csv(path,index=False)
class DS(torch.utils.data.Dataset):
    def __init__(self,texts,labels,tok,ml=128):
        self.enc=tok(list(texts),truncation=True,padding="max_length",max_length=ml,return_tensors="pt")
        self.labels=torch.tensor(list(labels),dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        it={k:v[i] for k,v in self.enc.items()}; it["labels"]=self.labels[i]; return it

TASK="ben_sarc_binary"; LABEL_COL="label_binary"; NUM_LABELS=2
def load_task():
    def rd(sp):
        df=pd.read_csv(SPLITS/f"{TASK}_{sp}.csv")[["text",LABEL_COL]].dropna(subset=["text"]).copy()
        df["text"]=df["text"].astype(str); df[LABEL_COL]=df[LABEL_COL].astype(int); return df
    tr,va,te=rd("train"),rd("val"),rd("test"); assert_no_leak(tr,va,te); return tr,va,te

ROOT: /workspace/Sarcasm_detection | SPLITS exists: True


In [12]:
# ---- adversarial + lever utilities (shared with NB09) ----
class FGM:
    def __init__(self,model,eps=0.5,emb="word_embeddings"):
        self.m=model; self.eps=eps; self.emb=emb; self.bk={}
    def attack(self):
        for n,p in self.m.named_parameters():
            if p.requires_grad and self.emb in n and p.grad is not None:
                self.bk[n]=p.data.clone(); nm=torch.norm(p.grad)
                if nm!=0 and not torch.isnan(nm): p.data.add_(self.eps*p.grad/nm)
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.bk: p.data=self.bk[n]
        self.bk={}
class AWP:
    def __init__(self,model,adv_lr=1.0,adv_eps=1e-3,param="weight"):
        self.m=model; self.lr=adv_lr; self.eps=adv_eps; self.param=param; self.bk={}; self.be={}
    def perturb(self):
        e=1e-6
        for n,p in self.m.named_parameters():
            if p.requires_grad and p.grad is not None and self.param in n:
                self.bk[n]=p.data.clone(); ge=self.eps*p.abs().detach()
                self.be[n]=(self.bk[n]-ge,self.bk[n]+ge)
                ng=torch.norm(p.grad); nw=torch.norm(p.data.detach())
                if ng!=0 and not torch.isnan(ng):
                    p.data.add_(self.lr*p.grad*(nw+e)/(ng+e))
                    p.data=torch.min(torch.max(p.data,self.be[n][0]),self.be[n][1])
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.bk: p.data=self.bk[n]
        self.bk={}; self.be={}
def get_base(m): return m.base_model
def freeze_lower(model,n_freeze,freeze_emb=False):
    base=get_base(model)
    if freeze_emb:
        for p in base.embeddings.parameters(): p.requires_grad=False
    for i in range(min(n_freeze,len(base.encoder.layer))):
        for p in base.encoder.layer[i].parameters(): p.requires_grad=False
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
def llrd_groups(model,base_lr,head_lr,decay=0.95,wd=0.01):
    base=get_base(model); L=len(base.encoder.layer); nod=("bias","LayerNorm.weight","layer_norm.weight")
    g=[]; seen=set()
    def add(named,lr):
        dp=[p for n,p in named if p.requires_grad and not any(x in n for x in nod)]
        nd=[p for n,p in named if p.requires_grad and any(x in n for x in nod)]
        if dp: g.append({"params":dp,"lr":lr,"weight_decay":wd})
        if nd: g.append({"params":nd,"lr":lr,"weight_decay":0.0})
        for _,p in named: seen.add(id(p))
    add(list(base.embeddings.named_parameters()), base_lr*(decay**(L+1)))
    for i,layer in enumerate(base.encoder.layer): add(list(layer.named_parameters()), base_lr*(decay**(L-i)))
    add([(n,p) for n,p in model.named_parameters() if id(p) not in seen], head_lr)
    return g
class AdvTrainer(Trainer):
    def __init__(self,*a,adv=None,fgm_eps=0.5,awp_eps=1e-3,rdrop=0.0,llrd=False,
                 llrd_decay=0.95,head_lr=1e-4,base_lr=2e-5,wd=0.01,**k):
        super().__init__(*a,**k); self.adv=adv; self.rdrop=rdrop; self.llrd=llrd
        self.llrd_decay=llrd_decay; self.head_lr=head_lr; self.base_lr=base_lr; self._wd=wd
        self._fgm=FGM(self.model,fgm_eps) if adv in ("fgm","fgm_awp") else None
        self._awp=AWP(self.model,adv_eps=awp_eps) if adv in ("awp","fgm_awp") else None
    def create_optimizer(self):
        if self.optimizer is None and self.llrd:
            self.optimizer=torch.optim.AdamW(llrd_groups(self.model,self.base_lr,self.head_lr,self.llrd_decay,self._wd))
            return self.optimizer
        return super().create_optimizer()
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        if self.rdrop>0:
            y=inputs["labels"]; o1=model(**inputs); o2=model(**inputs)
            ce=(F.cross_entropy(o1.logits,y)+F.cross_entropy(o2.logits,y))/2
            p=F.log_softmax(o1.logits,-1); q=F.log_softmax(o2.logits,-1)
            kl=(F.kl_div(p,q.exp(),reduction="batchmean")+F.kl_div(q,p.exp(),reduction="batchmean"))/2
            loss=ce+self.rdrop*kl; return (loss,o1) if return_outputs else loss
        return super().compute_loss(model,inputs,return_outputs)
    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, dict(inputs))

        if self.args.n_gpu > 1:
            loss = loss.mean()

        self.accelerator.backward(loss)

        if self._fgm is not None:
            self._fgm.attack()

            with self.compute_loss_context_manager():
                la = self.compute_loss(model, dict(inputs))

            if self.args.n_gpu > 1:
                la = la.mean()

            self.accelerator.backward(la)
            self._fgm.restore()

        if self._awp is not None:
            self._awp.perturb()

            with self.compute_loss_context_manager():
                lw = self.compute_loss(model, dict(inputs))

            if self.args.n_gpu > 1:
                lw = lw.mean()

            self.accelerator.backward(lw)
            self._awp.restore()

        return loss.detach()
def build_args(out,epochs,batch,lr,scheduler="linear",warmup=0.1,wd=0.01,ls=0.0,
               adv=False,grad_clip=1.0):
    # adversarial double-backward is unstable under fp16 grad-scaling -> use bf16 if available else fp32
    bf16 = HAS_BF16
    fp16 = (HAS_CUDA and not HAS_BF16 and not adv)
    common=dict(output_dir=str(out),num_train_epochs=epochs,per_device_train_batch_size=batch,
                per_device_eval_batch_size=batch*2,learning_rate=lr,weight_decay=wd,warmup_ratio=warmup,
                lr_scheduler_type=scheduler,label_smoothing_factor=ls,max_grad_norm=grad_clip,
                save_strategy="epoch",logging_strategy="epoch",save_total_limit=1,
                load_best_model_at_end=True,metric_for_best_model="macro_f1",greater_is_better=True,
                report_to="none",seed=SEED,data_seed=SEED,fp16=fp16,bf16=bf16)
    try: return TrainingArguments(evaluation_strategy="epoch",**common)
    except TypeError: return TrainingArguments(eval_strategy="epoch",**common)

In [13]:
# ============================================================
#  STAGE-A PIPELINES — fixed comparison budget (EPOCHS_A), FGM eps=0.5 where used
# ============================================================
EPOCHS_A=8; BATCH=32; BB="csebuetnlp/banglabert"; BB_LARGE="csebuetnlp/banglabert_large"

PIPELINES = [   # fair set (same budget) -> winner chosen by VAL macro-F1
 dict(name="P0_fullft",            model=BB, epochs=EPOCHS_A, lr=2e-5, max_len=128),
 dict(name="P1_fullft_fgm",        model=BB, epochs=EPOCHS_A, lr=2e-5, max_len=128, adv="fgm"),
 dict(name="P2_fullft_llrd",       model=BB, epochs=EPOCHS_A, lr=2e-5, max_len=128, llrd=True),
 dict(name="P3_fullft_fgm_llrd",   model=BB, epochs=EPOCHS_A, lr=2e-5, max_len=128, adv="fgm", llrd=True),
 dict(name="P4_freeze6_fgm",       model=BB, epochs=EPOCHS_A, lr=2e-5, max_len=128, adv="fgm", freeze=6),
 dict(name="P5_freeze6_fgm_llrd",  model=BB, epochs=EPOCHS_A, lr=2e-5, max_len=128, adv="fgm", freeze=6, llrd=True),
 dict(name="P6_maxlen192_fgm_llrd",model=BB, epochs=EPOCHS_A, lr=2e-5, max_len=192, adv="fgm", llrd=True),
 dict(name="P7_fgm_llrd_ls",       model=BB, epochs=EPOCHS_A, lr=2e-5, max_len=128, adv="fgm", llrd=True, ls=0.1),
 dict(name="P8_large_fgm_llrd",    model=BB_LARGE, epochs=EPOCHS_A, lr=1e-5, max_len=128, adv="fgm", llrd=True, batch=16),
]
# faithful competitor reproduction (NOT a fair-budget row; measures leakage gap vs their 0.890)
ZIHAN = dict(name="ZIHAN_repro", model=BB, epochs=15, lr=2e-5, max_len=128, adv="fgm",
             freeze=6, llrd=True, ls=0.1, scheduler="cosine", reference=0.890, faithful=True)

ALL_CONFIGS = [
    dict(
        name="P7_fgm_llrd_ls",
        model=BB,
        epochs=EPOCHS_A,
        lr=2e-5,
        max_len=128,
        adv="fgm",
        llrd=True,
        ls=0.1,
        grad_clip=0.5,
    ),
    dict(
        name="ZIHAN_repro",
        model=BB,
        epochs=15,
        lr=2e-5,
        max_len=128,
        adv="fgm",
        freeze=6,
        llrd=True,
        ls=0.1,
        scheduler="cosine",
        grad_clip=0.5,
        reference=0.890,
        faithful=True,
    ),
]
tok_cache={}
print(f"{len(PIPELINES)} fair pipelines + 1 competitor reproduction")

9 fair pipelines + 1 competitor reproduction


In [14]:
def run_pipeline(cfg):
    set_seed(SEED); tr,va,te=load_task()
    mn=cfg["model"]; ml=cfg.get("max_len",128); batch=cfg.get("batch",BATCH)
    if mn not in tok_cache: tok_cache[mn]=AutoTokenizer.from_pretrained(mn)
    tok=tok_cache[mn]
    tr_ds=DS(tr["text"],tr[LABEL_COL],tok,ml); va_ds=DS(va["text"],va[LABEL_COL],tok,ml); te_ds=DS(te["text"],te[LABEL_COL],tok,ml)
    out=CKPT/f"08_{cfg['name']}"
    if out.exists(): shutil.rmtree(out,ignore_errors=True)
    model=AutoModelForSequenceClassification.from_pretrained(mn,num_labels=NUM_LABELS)
    n_train=sum(p.numel() for p in model.parameters() if p.requires_grad)
    if cfg.get("freeze",0): n_train=freeze_lower(model,cfg["freeze"],freeze_emb=False)
    args=build_args(out,cfg["epochs"],batch,cfg["lr"],scheduler=cfg.get("scheduler","linear"),
                    ls=cfg.get("ls",0.0),adv=bool(cfg.get("adv")))
    trn=AdvTrainer(model=model,args=args,train_dataset=tr_ds,eval_dataset=va_ds,
                   compute_metrics=make_compute_metrics(),
                   callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
                   adv=cfg.get("adv"),llrd=cfg.get("llrd",False),base_lr=cfg["lr"])
    t0=time.time(); trn.train(); secs=round(time.time()-t0,1)
    vlog=trn.predict(va_ds).predictions; tlog=trn.predict(te_ds).predictions
    vm,_=eval_from_logits(va[LABEL_COL].values,vlog,NUM_LABELS)
    tm,tp=eval_from_logits(te[LABEL_COL].values,tlog,NUM_LABELS)
    save_predictions(PRED/f"08_{cfg['name']}_{TASK}_val_predictions.csv", va["text"].values,va[LABEL_COL].values,vlog,NUM_LABELS,extra={"config":cfg["name"],"split":"val"})
    save_predictions(PRED/f"08_{cfg['name']}_{TASK}_test_predictions.csv",te["text"].values,te[LABEL_COL].values,tlog,NUM_LABELS,extra={"config":cfg["name"],"split":"test"})
    json.dump({"test":confusion_matrix(te[LABEL_COL].values,tp).tolist()},open(TABLES/f"08_{cfg['name']}_confusion.json","w"),indent=2)
    row={"config":cfg["name"],"model":mn,"epochs":cfg["epochs"],"max_len":ml,"lr":cfg["lr"],"batch":batch,
         "adv":cfg.get("adv"),"freeze":cfg.get("freeze",0),"llrd":cfg.get("llrd",False),"ls":cfg.get("ls",0.0),
         "scheduler":cfg.get("scheduler","linear"),"faithful_repro":cfg.get("faithful",False),
         "trainable_params":int(n_train),"time_seconds":secs,
         "val_macro_f1":vm["macro_f1"],"test_macro_f1":tm["macro_f1"],
         "val_accuracy":vm["accuracy"],"test_accuracy":tm["accuracy"]}
    del model,trn; gc.collect()
    if HAS_CUDA: torch.cuda.empty_cache()
    shutil.rmtree(out,ignore_errors=True)
    return row

In [15]:
SUMMARY=TABLES/"08_pipeline_search.csv"
rows=[]; done=set()
if SUMMARY.exists():
    prev=pd.read_csv(SUMMARY); rows=prev.to_dict("records"); done=set(prev["config"]); print("resuming, done:",len(done))
for cfg in ALL_CONFIGS:
    if cfg["name"] in done: print("skip:",cfg["name"]); continue
    print("\n"+"="*72+f"\n[{cfg['name']}] {'(faithful repro)' if cfg.get('faithful') else ''}\n"+"="*72)
    try:
        rows.append(run_pipeline(cfg)); pd.DataFrame(rows).to_csv(SUMMARY,index=False)
        r=rows[-1]; print(f"  -> val={r['val_macro_f1']:.4f}  test={r['test_macro_f1']:.4f}  ({r['time_seconds']}s)")
    except Exception as e:
        print(f"  !! FAILED {cfg['name']}: {type(e).__name__}: {e}")
        gc.collect();  torch.cuda.empty_cache() if HAS_CUDA else None
print("\nDONE ->", SUMMARY.name)


[P7_fgm_llrd_ls] 


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.578300,0.500521,0.792629
2,0.477600,0.500151,0.799702
3,0.393700,0.509739,0.804234
4,0.314100,0.537258,0.800929
5,0.264800,0.569923,0.795917
6,0.238800,0.610383,0.793544


  -> val=0.8042  test=0.8003  (630.1s)

[ZIHAN_repro] (faithful repro)


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.603900,0.522390,0.776423
2,0.517400,0.516356,0.785136
3,0.462500,0.504754,0.784151
4,0.394100,0.506285,0.785082
5,0.323700,0.548560,0.791366
6,0.274700,0.601686,0.787459
7,0.247000,0.625355,0.781687
8,0.229300,0.654041,0.781766


  -> val=0.7914  test=0.7828  (738.2s)

DONE -> 08_pipeline_search.csv


In [16]:
# ---- choose best fair pipeline (by VAL); report protocol/reproduction gap for Zihan repro ----
df=pd.read_csv(SUMMARY)
fair=df[df["faithful_repro"]!=True].sort_values("val_macro_f1",ascending=False)
print("="*92); print("  STAGE-A PIPELINE SEARCH — ranked by VAL macro-F1 (fair, fixed-budget set)"); print("="*92)
print(fair[["config","val_macro_f1","test_macro_f1","trainable_params","time_seconds"]].to_string(index=False,float_format="%.4f"))
best=fair.iloc[0]
print(f"\nBEST PIPELINE: {best['config']}  (val={best['val_macro_f1']:.4f}, test={best['test_macro_f1']:.4f})")
json.dump({"best_pipeline":best["config"],"val_macro_f1":float(best["val_macro_f1"]),
           "test_macro_f1":float(best["test_macro_f1"])}, open(TABLES/"08_best_pipeline.json","w"), indent=2)
print("-> saved 08_best_pipeline.json (consumed by NB09)")

z=df[df["config"]=="ZIHAN_repro"]
if len(z):
    zt=float(z.iloc[0]["test_macro_f1"])
    print("\n"+"="*92); print("  COMPETITOR REPRODUCTION — Zihan et al. recipe on OUR clean splits"); print("="*92)
    print(f"  Zihan reported (leaky, no dedup): 0.890")
    print(f"  Same recipe on clean splits     : {zt:.4f}")
    print(f"  LEAKAGE GAP                      : {0.890-zt:+.4f}")
    print("  -> If the gap is large and positive, their SOTA does not survive leakage control (headline finding).")

  STAGE-A PIPELINE SEARCH — ranked by VAL macro-F1 (fair, fixed-budget set)
        config  val_macro_f1  test_macro_f1  trainable_params  time_seconds
P7_fgm_llrd_ls        0.8042         0.8003         110618882      630.1000

BEST PIPELINE: P7_fgm_llrd_ls  (val=0.8042, test=0.8003)
-> saved 08_best_pipeline.json (consumed by NB09)

  COMPETITOR REPRODUCTION — Zihan et al. recipe on OUR clean splits
  Zihan reported (leaky, no dedup): 0.890
  Same recipe on clean splits     : 0.7828
  LEAKAGE GAP                      : +0.1072
  -> If the gap is large and positive, their SOTA does not survive leakage control (headline finding).
